# Class 24 — LLM Observability
### OpenTelemetry concepts → Langfuse as the concrete backend

This notebook builds a small self-contained ReAct agent, instruments it three ways,
opens the trace in the Langfuse UI, attaches an eval score to a live trace, then
**breaks it on purpose** (the async-flush gotcha) and fixes it.

**Stack:** OpenRouter (agent) + Langfuse Cloud free tier (backend). No GPU, no Docker.

> Concepts first: a **trace** = one request's lifecycle, a **span** = one operation inside it,
> a **generation** = a span that wrapped a model call (model / prompt / completion / tokens).
> Langfuse's core component is literally a `LangfuseSpanProcessor` — an OpenTelemetry span processor.
>
> *(Written against the Langfuse v4 Python SDK — current as of mid-2026.)*


## Part 0 — Setup

In [1]:
# Install. Langfuse v4 SDK + OpenAI client (used as an OpenRouter-compatible client).
!pip install -q langfuse openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 609.2/609.2 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 8.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.3.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.43.0 which is incompatible.
google-adk 2.3.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.43.0 which is incompatible.


In [17]:

import os

# --- Langfuse ---
os.environ["LANGFUSE_PUBLIC_KEY"] = "REDACTED-SECRET-REMOVED"
os.environ["LANGFUSE_SECRET_KEY"] = "REDACTED-SECRET-REMOVED"

os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"

# --- OpenRouter (agent's model provider) ---
os.environ["OPENROUTER_API_KEY"] = "REDACTED-SECRET-REMOVED"

MODEL = "google/gemma-4-26b-a4b-it:free"  # swap if rate-limited


In [18]:
# Verify Langfuse can authenticate.
from langfuse import get_client

langfuse = get_client()
assert langfuse.auth_check(), "Auth failed — check keys and that LANGFUSE_HOST region matches your project."
print("Langfuse connected.")


Langfuse connected.


## Part 1 — A self-contained ReAct agent (no tracing yet)

A tiny agent with two tools. This is the system we'll observe. Run it once and notice:
you can see the *final answer*, but you cannot see the cost, the latency per step, or the
internal tree of decisions. `print()` is all you have.

In [19]:
from openai import OpenAI
import json as _json

# OpenRouter is OpenAI-API-compatible -> reuse the OpenAI client, point base_url at it.
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

# --- Two trivial tools ---
def get_word_length(word: str) -> int:
    return len(word)

def multiply(a: float, b: float) -> float:
    return a * b

TOOLS = {
    "get_word_length": get_word_length,
    "multiply": multiply,
}

TOOL_SPEC = """You have these tools. To call one, output EXACTLY one line:
ACTION: <tool_name> <json_args>
Examples:
ACTION: get_word_length {"word": "observability"}
ACTION: multiply {"a": 6, "b": 7}
When you have the final answer, output:
FINAL: <answer>
Tools:
- get_word_length(word): returns integer length
- multiply(a, b): returns a*b
"""


In [21]:
def call_model(messages):
    resp = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=300)
    return resp.choices[0].message.content

def run_agent(question, max_steps=5):
    messages = [
        {"role": "system", "content": TOOL_SPEC},
        {"role": "user", "content": question},
    ]
    for step in range(max_steps):
        out = call_model(messages)
        print(f"[step {step}] model: {out.strip()[:120]}")
        if "FINAL:" in out:
            return out.split("FINAL:")[-1].strip()
        if "ACTION:" in out:
            line = [l for l in out.splitlines() if l.strip().startswith("ACTION:")][0]
            rest = line.split("ACTION:")[-1].strip()
            name, _, arg_str = rest.partition(" ")
            args = _json.loads(arg_str)
            result = TOOLS[name](**args)
            print(f"[step {step}] tool {name}{args} -> {result}")
            messages.append({"role": "assistant", "content": out})
            messages.append({"role": "user", "content": f"OBSERVATION: {result}"})
        else:
            messages.append({"role": "assistant", "content": out})
            messages.append({"role": "user", "content": "Use ACTION: or FINAL:"})
    return "(no final answer)"

print(run_agent("How long is the word 'observability', and what is that number times 3?"))


[step 0] model: <|tool_call>call:get_word_length {"word": "observability"}<tool_call|>
[step 1] model: ACTION: get_word_length {"word": "observability"}
[step 1] tool get_word_length{'word': 'observability'} -> 13


AttributeError: 'NoneType' object has no attribute 'strip'

**Felt problem.** You saw the final answer. But:
- How many tokens did this cost?
- Which step was the slow one?
- If step 3 had returned garbage, where would you even look?

`print()` gives you a flat scroll, not a causal tree. Let's fix that.

## Part 2 — Raw OpenTelemetry: structured + causal, but unreadable for LLM work

Before reaching for any product, let's instrument with **plain OpenTelemetry** and a console
exporter — no backend, no Docker. This is the standard itself, nothing on top. We build a span
tree by hand and print the raw spans, so you see exactly what OTel gives you... and what it doesn't.

In [12]:
!pip install -q opentelemetry-sdk opentelemetry-api


In [13]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

# Build the OTel pipeline from slide 13, by hand:
#   tracer -> span processor -> exporter (here: just print to console)
otel_provider = TracerProvider()
otel_provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(otel_provider)
tracer = trace.get_tracer("class24")

print("Raw OTel pipeline ready. Spans will print as JSON below.")


Raw OTel pipeline ready. Spans will print as JSON below.


In [14]:
# Manually trace one agent step as raw OTel spans.
# A "generation" in OTel is just a span with conventionally-named attributes.
with tracer.start_as_current_span("agent-run") as root:
    root.set_attribute("user", "student")
    with tracer.start_as_current_span("llm-call") as gen:
        gen.set_attribute("gen_ai.request.model", MODEL)
        gen.set_attribute("gen_ai.usage.input_tokens", 412)   # we'd have to count these ourselves
        gen.set_attribute("gen_ai.usage.output_tokens", 87)
        gen.set_attribute("llm.prompt", "How long is 'observability' x3?")
        gen.set_attribute("llm.completion", "13 letters; 13 x 3 = 39.")
# Spans print on exit. Read the output carefully.


**What you just saw — read your own output:**

- It IS structured. It IS causal — but the parent link is two raw hex IDs (`parent_id` ->
  `span_id`) you'd match by eye.
- Tokens are there as `gen_ai.usage.input_tokens: 412` — **but you had to count them yourself**,
  and there's **no cost anywhere** (OTel has no model price table).
- The prompt/completion are flat strings in an attributes dict — **no chat rendering**.
- There is **no field for a quality score**, no session grouping, no UI.

This is the gap. Raw OTel is the right *substrate*, but reading an LLM agent's behavior from
JSON blobs is unworkable. That gap is the entire reason LLM-observability products exist.

## Part 3 — Why providers exist: the LLM-aware layer on top of OTel

| What you need | Raw OTel | LLM-obs provider |
|---|---|---|
| Span tree, OTLP wire format | yes | yes (built on it) |
| Token usage stored | yes (manual) | yes (auto-captured) |
| **Cost in dollars** | no | yes (price tables) |
| **Prompt/completion rendered as chat** | no | yes |
| **Attach eval scores** | no | yes |
| **Sessions, users, dashboards, UI** | no | yes |

Providers (Langfuse, LangSmith, Phoenix, Helicone) are *not* alternatives to OTel — most are
**OTel backends**. They consume the same spans and add the LLM-aware storage and UI. We'll use
**Langfuse**: open-source, OTel-native (its core is literally a `LangfuseSpanProcessor`), broad
framework support. The concepts transfer to any of them.

## Part 4 — Langfuse: instrument the agent (least -> most control)

Three ways.

### 4a — Drop-in: trace model calls with a one-line import swap
`from langfuse.openai import openai` returns an OpenAI-compatible client whose every call
is automatically captured as a **generation** (model / prompt / completion / tokens) — the
thing we had to fake by hand in raw OTel.

In [15]:
from langfuse.openai import OpenAI as LFOpenAI

# Same constructor, now auto-traced.
client = LFOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
print("Model calls are now auto-traced as generations.")


Model calls are now auto-traced as generations.


### 4b — Decorator: trace *your own* logic so calls nest correctly
`@observe()` turns a function into a span. Anything it calls nests underneath, producing the
causal tree. We re-run the agent under a single parent span.

In [16]:
from langfuse import observe

@observe()
def agent_step(messages, step):
    out = call_model(messages)   # auto-traced generation nests under this span
    return out

@observe()
def traced_agent(question, max_steps=5):
    messages = [
        {"role": "system", "content": TOOL_SPEC},
        {"role": "user", "content": question},
    ]
    for step in range(max_steps):
        out = agent_step(messages, step)
        if "FINAL:" in out:
            return out.split("FINAL:")[-1].strip()
        if "ACTION:" in out:
            line = [l for l in out.splitlines() if l.strip().startswith("ACTION:")][0]
            rest = line.split("ACTION:")[-1].strip()
            name, _, arg_str = rest.partition(" ")
            args = _json.loads(arg_str)
            result = TOOLS[name](**args)
            messages.append({"role": "assistant", "content": out})
            messages.append({"role": "user", "content": f"OBSERVATION: {result}"})
        else:
            messages.append({"role": "assistant", "content": out})
            messages.append({"role": "user", "content": "Use ACTION: or FINAL:"})
    return "(no final answer)"

answer = traced_agent("How long is the word 'observability', and what is that number times 3?")
langfuse.flush()   # short-lived process -> force the batch out (more on this in Part 5)
print("Answer:", answer)
print("Open https://cloud.langfuse.com -> your project -> Traces. Click the latest trace.")


Answer: The word 'observability' has 13 letters, and 13 times 3 equals 39
Open https://cloud.langfuse.com -> your project -> Traces. Click the latest trace.


### 4c — Context manager: maximum control over span boundaries + attributes
`start_as_current_observation` lets you decide exactly where a span/generation starts and ends,
its type, and what attributes (tags, user, session) hang on it.

In [ ]:
with langfuse.start_as_current_observation(as_type="span", name="manual-demo") as span:
    span.update(input={"task": "demo"})
    with langfuse.start_as_current_observation(
        as_type="generation", name="llm-call", model=MODEL
    ) as gen:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": "Say 'traced' and nothing else."}],
            max_tokens=10,
        )
        gen.update(output=resp.choices[0].message.content)
    span.update(output="done")

langfuse.flush()
print("Manual trace sent. Note how the generation nests under the span in the UI.")


In the UI you now see the tree: parent span → nested generation, each with latency,
token counts, and dollar cost. The three questions from Part 1 are now answerable by *looking*.

## Part 5 — Close the loop: attach an eval score to a live trace

The evals class scored a *static* dataset. Here we score a *real run*. We grade the agent's
output with an LLM-as-judge and write the score back onto the trace.

In [ ]:
# Capture the trace id of a fresh run so we can attach a score to it.
with langfuse.start_as_current_observation(as_type="span", name="scored-run") as root:
    answer = traced_agent("What is the length of 'trace' multiplied by 10?")
    trace_id = langfuse.get_current_trace_id()
    root.update(output=answer)

langfuse.flush()
print("answer:", answer, "| trace:", trace_id)


In [ ]:
# Simple LLM-as-judge: does the answer correctly reason about the task?
judge_prompt = f"""You are grading an agent answer. Question involved len('trace')=5 times 10 = 50.
Agent answer: {answer!r}
Reply with ONLY a number 1 if the answer is 50 (or clearly states 50), else 0."""

judge = client.chat.completions.create(
    model=MODEL, messages=[{"role": "user", "content": judge_prompt}], max_tokens=5
)
raw = judge.choices[0].message.content.strip()
score_value = 1.0 if "1" in raw else 0.0

langfuse.create_score(
    trace_id=trace_id,
    name="judge_correct",
    value=score_value,
    comment=f"LLM judge raw output: {raw!r}",
)
langfuse.flush()
print(f"Attached score judge_correct={score_value} to trace {trace_id}")
print("Refresh the trace in the UI — the score now appears on it.")


Do this across real traffic and the dashboard gives you a **live** quality signal.
A prompt regression shows up as the score line dropping *today*, not next release.

## Part 6 — Sessions & environments (do this from day one)

`session_id` groups multiple traces into one conversation. `environment` keeps test runs out
of your production dashboards.

In [ ]:
# Group two turns of a conversation under one session, tagged as a non-prod environment.
# v4: trace-level fields (session_id, tags, user_id, environment) are set with
# propagate_attributes() as a context manager — everything traced inside inherits them.
from langfuse import propagate_attributes
import uuid
session_id = f"demo-session-{uuid.uuid4().hex[:8]}"

for turn in ["How long is 'span'?", "Multiply that by 4."]:
    with propagate_attributes(session_id=session_id, tags=["class24", "env:dev"]):
        with langfuse.start_as_current_observation(as_type="span", name="chat-turn") as s:
            ans = traced_agent(turn)
            s.update(output=ans)

langfuse.flush()
print(f"Two turns grouped under session {session_id}. See Sessions view in the UI.")


## Part 7 — BREAK IT ON PURPOSE: the async-flush gotcha

The single most common "my traces vanished" bug. Tracing is **async + batched** — that's why
it adds ~zero latency in production. But a short-lived process (a notebook cell, a script, a
serverless function) can **end before the background batch flushes**. The data dies in the queue.

In [ ]:
# THE BREAK: run, then immediately "end" without flushing.
# In a real script the process would exit here. We simulate by NOT calling flush
# and creating a brand-new client so nothing else triggers a flush.
from langfuse import get_client as _gc
_lf = _gc()

with _lf.start_as_current_observation(as_type="span", name="unflushed-run") as s:
    s.update(input={"q": "this trace may never arrive"}, output="done")

# (no flush)  -> check the UI now. The 'unflushed-run' trace is likely missing/partial.
print("Created 'unflushed-run' WITHOUT flushing. Check the UI — likely not there yet.")


In [ ]:
# THE FIX: one line. Forces the queue to send and blocks until done.
_lf.flush()
print("Flushed. 'unflushed-run' now appears in the UI.")
print("Lesson: ANY OTel-based tracing batches by default ->")
print("short-lived processes need an explicit flush() / shutdown().")


### Takeaways
- **trace / span / generation** are the vocabulary; **OTel** is the substrate every serious tool sits on.
- Langfuse = an OTLP-compatible backend with an LLM-aware UI; `LangfuseSpanProcessor` *is* an OTel span processor.
- Observability + evals = a feedback loop that catches **silent** regressions before users do.
- Short-lived processes must `flush()`.
